# Clothes segmentation: two improvement experiments

**A. Multi-scale averaging (cell 5).** No training. The final model predicts each photo at three sizes and mirrored, and the answers are averaged. About 15 minutes. Works on a T4.

**B. Better starting weights (cell 6).** Trains SegFormer-B3 for 12 epochs starting from weights that were already trained for segmentation on ADE20K, and compares it with the 12-epoch model that started from ImageNet weights. About 2.5 hours. Needs the L4.

Both decisions are made on the validation split, by rules written down in PLAN.md, section 24, before the results exist.

## 1. Settings

In [1]:
REPO_URL = ""

DRIVE_ROOT = "/content/drive/MyDrive/clothseg"
CODE_DIR = "/content/clothes-segmentation"
CODE_VERSION = "10"
HEIGHT, WIDTH = 576, 384
SESSION_BUDGET_MIN = 600

## 2. Mount Drive and show the environment

In [2]:
import os, platform, shutil, time
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
SESSION_START = time.time()

def minutes_left():
    """Minutes of the session budget that remain; passed to every training command."""
    return max(1, SESSION_BUDGET_MIN - (time.time() - SESSION_START) / 60)

import torch, transformers
print("python      ", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: change the runtime type to a GPU")
if torch.cuda.is_available():
    print("GPU memory  ", f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("CPU cores   ", os.cpu_count())
print("local disk  ", f"{shutil.disk_usage('/content').free / 1024**3:.0f} GB free")
print("Drive       ", f"{shutil.disk_usage(DRIVE_ROOT).free / 1024**3:.0f} GB free")
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > L4 GPU, then run again."

Mounted at /content/drive
python       3.13.15 | torch 2.11.0+cu130 | transformers 5.18.0
GPU          NVIDIA L4
GPU memory   22.0 GB
CPU cores    12
local disk   193 GB free
Drive        184 GB free


## 3. Get the code

In [3]:
import subprocess, zipfile

def code_version(folder):
    path = os.path.join(folder, "VERSION")
    return open(path).read().strip() if os.path.exists(path) else "1"

os.chdir("/content")
if REPO_URL:
    if os.path.isdir(f"{CODE_DIR}/.git"):
        print(subprocess.run(["git", "-C", CODE_DIR, "pull"], capture_output=True, text=True).stdout)
    else:
        r = subprocess.run(["git", "clone", REPO_URL, CODE_DIR], capture_output=True, text=True)
        assert r.returncode == 0, f"git clone failed. Is the URL correct and the repository public?\n{r.stderr}"
elif os.path.isdir(f"{DRIVE_ROOT}/code") and code_version(f"{DRIVE_ROOT}/code") == CODE_VERSION:
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    shutil.copytree(f"{DRIVE_ROOT}/code", CODE_DIR)
    print(f"code version {CODE_VERSION} copied from Drive")
else:
    from google.colab import files
    if os.path.exists("clothseg_code.zip"):
        os.remove("clothseg_code.zip")   # otherwise Colab renames the new upload
    print(f"Drive has code version {code_version(f'{DRIVE_ROOT}/code')}, this notebook needs version {CODE_VERSION}.")
    print("Choose the NEW clothseg_code.zip from the project folder on your computer.")
    uploaded = files.upload()
    assert "clothseg_code.zip" in uploaded, f"Expected a file named clothseg_code.zip, got {list(uploaded)}."
    for folder in (CODE_DIR, f"{DRIVE_ROOT}/code"):   # the Drive copy is used by later sessions
        shutil.rmtree(folder, ignore_errors=True)
        with zipfile.ZipFile("clothseg_code.zip") as z:
            z.extractall(folder)
    print("code uploaded and saved to Drive")

assert code_version(CODE_DIR) == CODE_VERSION, (
    f"The code is version {code_version(CODE_DIR)} but this notebook needs version {CODE_VERSION}. "
    "You uploaded an old clothseg_code.zip. Run this cell again and choose the new one.")
for needed in ("scripts/train.py", "scripts/resume_test.py", "src/clothseg/engine.py", "requirements.txt"):
    assert os.path.exists(f"{CODE_DIR}/{needed}"), f"{needed} is missing from {CODE_DIR}. Check the upload or the repository."
os.chdir(CODE_DIR)
print("code ready in", os.getcwd(), "| version", code_version(CODE_DIR))

Drive has code version 9, this notebook needs version 10.
Choose the NEW clothseg_code.zip from the project folder on your computer.


Saving clothseg_code.zip to clothseg_code.zip
code uploaded and saved to Drive
code ready in /content/clothes-segmentation | version 10


## 4. Install requirements and prepare the dataset

In [4]:
!pip install -q -r requirements.txt
import segmentation_models_pytorch, timm, cv2
print("segmentation_models_pytorch", segmentation_models_pytorch.__version__, "| timm", timm.__version__, "| opencv", cv2.__version__)
!python scripts/prepare_data.py --drive-root "{DRIVE_ROOT}" --height {HEIGHT} --width {WIDTH}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 13.5 MB/s eta 0:00:00
segmentation_models_pytorch 0.5.0 | timm 1.0.30 | opencv 4.14.0
[prepare] cache name: atr_576x384 | local: /content/data/atr_576x384 | Drive archive: /content/drive/MyDrive/clothseg/data/cache/atr_576x384.tar
[prepare] free local disk: 193.3 GB
[prepare] found archive on Drive (1.6 GB). Copying and extracting...
[storage] copied atr_576x384.tar (1.6 GB) -> /content/data in 23s
[storage] extracted atr_576x384.tar -> /content/data in 13s
[prepare] done in 38s
[splits] moved 56 val/test near-duplicates of training images into train. New sizes: train=16038, val=690, test=960


## 5. Experiment A: multi-scale averaging on the final model

In [5]:
import json, subprocess, sys

RUN_NAME = "final_segformer_b3_e30"
CHECKPOINT = f"{DRIVE_ROOT}/runs/{RUN_NAME}/checkpoints/best.pt"
SCALES = ["0.75", "1.0", "1.25"]
MIN_GAIN = 0.002      # multi-scale is adopted if it adds at least 0.2 points of validation mIoU over flip alone

os.chdir(CODE_DIR)
assert os.path.exists(CHECKPOINT), f"No checkpoint at {CHECKPOINT}."

def evaluate(split, tag, extra):
    out = f"{DRIVE_ROOT}/reports/eval_{RUN_NAME}_{split}_{tag}"
    print(f"evaluating {split}, {tag} ...", flush=True)
    r = subprocess.run([sys.executable, "scripts/evaluate.py", "--checkpoint", CHECKPOINT, "--split", split, "--out-dir", out] + extra,
                       capture_output=True, text=True)
    assert r.returncode == 0, "The evaluation failed. Send me this message: " + (r.stdout + r.stderr)[-3000:]
    return json.load(open(f"{out}/metrics.json"))

KEYS = ("miou", "garment_miou", "core_garment_miou", "clothing_binary_iou", "pixel_acc", "boundary_miou", "garment_boundary_miou", "ms_per_image")

def table(title, a, b):
    print()
    print(f"{title}  ({a['num_images']} images)")
    print(f"{'metric':<24}{'flip only':>11}{'multi-scale':>13}{'change':>9}")
    for k in KEYS:
        print(f"{k:<24}{a[k]:>11.4f}{b[k]:>13.4f}{b[k] - a[k]:>+9.4f}")

val_flip = evaluate("val", "flip2", ["--flip-tta"])
val_ms = evaluate("val", "multiscale", ["--flip-tta", "--scales"] + SCALES)
table("VALIDATION", val_flip, val_ms)

gain = val_ms["miou"] - val_flip["miou"]
adopted = gain >= MIN_GAIN
print()
print(f"multi-scale changes validation mIoU by {100 * gain:+.2f} points -> " + ("ADOPTED" if adopted else "NOT adopted") + " (rule: at least +0.20)")

if adopted:
    # The test split was already evaluated once with flip averaging. This is a second evaluation,
    # for a setting that was chosen on validation only.
    test_flip = json.load(open(f"{DRIVE_ROOT}/reports/eval_{RUN_NAME}_test_flip/metrics.json"))
    test_ms = evaluate("test", "multiscale", ["--flip-tta", "--scales"] + SCALES)
    table("TEST", test_flip, test_ms)
else:
    print("The test split is not evaluated again. The reported result stays as it is.")

evaluating val, flip2 ...
evaluating val, multiscale ...

VALIDATION  (690 images)
metric                    flip only  multi-scale   change
miou                         0.7888       0.7896  +0.0007
garment_miou                 0.7441       0.7447  +0.0006
core_garment_miou            0.7904       0.7866  -0.0037
clothing_binary_iou          0.9296       0.9300  +0.0004
pixel_acc                    0.9720       0.9720  -0.0000
boundary_miou                0.7349       0.7365  +0.0015
garment_boundary_miou        0.6808       0.6825  +0.0017
ms_per_image               110.9347     312.9871+202.0524

multi-scale changes validation mIoU by +0.07 points -> NOT adopted (rule: at least +0.20)
The test split is not evaluated again. The reported result stays as it is.


## 6. Experiment B: SegFormer-B3 started from ADE20K segmentation weights

In [6]:
ADE_RUN, BASE_RUN = "abl_segformer_b3_ade", "r1_segformer_b3"
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert gpu_gb >= 20, (f"This GPU has {gpu_gb:.0f} GB. Training needs the L4 (about 22 GB). "
                      "Choose Runtime > Change runtime type > L4 GPU, then run cells 1 to 4 and this cell again.")

os.chdir(CODE_DIR)
budget = int(minutes_left())
!python scripts/train.py --drive-root "{DRIVE_ROOT}" --run-name {ADE_RUN} --model segformer_b3_ade --height {HEIGHT} --width {WIDTH} --epochs 12 --time-budget-min {budget}

import pandas as pd
assert os.path.exists(f"{DRIVE_ROOT}/runs/{ADE_RUN}/val_log.csv"), "No epoch has finished yet. Read the training output above for the reason."
cols = ["garment_miou", "miou", "iou_upper-clothes", "iou_pants", "iou_skirt", "iou_dress", "iou_belt", "iou_scarf", "val_loss", "train_min"]
base = pd.read_csv(f"{DRIVE_ROOT}/runs/{BASE_RUN}/val_log.csv").set_index("epoch")
ade = pd.read_csv(f"{DRIVE_ROOT}/runs/{ADE_RUN}/val_log.csv").set_index("epoch")
print()
print("garment mIoU per epoch")
print(pd.DataFrame({"ImageNet start": base["garment_miou"], "ADE20K start": ade["garment_miou"]}).to_string())
last = min(len(base), len(ade))
print()
print(f"epoch {last}")
print(pd.DataFrame({"ImageNet start": base.loc[last, cols], "ADE20K start": ade.loc[last, cols],
                    "difference": ade.loc[last, cols] - base.loc[last, cols]}).round(4).to_string())

config.json: 100% 6.89k/6.89k [00:00<00:00, 16.3MB/s]

pytorch_model.bin: downloading bytes:  70% 133M/190M [00:02<00:00, 150MB/s, 9.27MB/s  ] 
pytorch_model.bin: downloading bytes:  89% 170M/190M [00:02<00:00, 160MB/s, 14.0MB/s  ]
pytorch_model.bin: downloading bytes: 100% 178M/178M [00:02<00:00, 66.7MB/s, 16.3MB/s  ]
pytorch_model.bin: reconstructing file: 100% 190M/190M [00:02<00:00, 70.8MB/s, 17.7MB/s  ]
Loading weights: 100% 644/644 [00:00<00:00, 22208.51it/s]
[model] segformer_b3_ade: 47.2 M parameters (encoder 44.1 M, head 3.2 M) | 15 classes | encoder and decoder: ADE20K segmentation weights loaded, classifier: new
[train] FRESH START in /content/drive/MyDrive/clothseg/runs/abl_segformer_b3_ade
[train] segformer_b3_ade | 15 classes | 576x384 | batch 16 | fp16 True | NVIDIA L4 | 8 workers
[train] data: train 16038, val 690 images (fingerprint 8b4d7a60ccb0) | 1002 steps/epoch
[train] schedule: 12 epochs = 12024 steps; this command runs until the end of epoch 12; time budget 591 m